# Phase 8 Lab — Reference Solution

**Phase:** Unsupervised Learning  
**Scenario:** A business wants actionable customer segments and an operations team wants a low-noise anomaly queue.

**Deliverable:** Validated segmentation profiles and anomaly rankings with stability, sensitivity, and actionability analysis.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Justify features, transformations, distance, and scaling.
2. Compare K-means, hierarchical, density, and mixture approaches.
3. Evaluate stability across seeds and resamples.
4. Profile clusters without using identifiers as features.
5. Compare statistical and Isolation Forest anomaly scores.
6. Evaluate alerts under a finite review budget.
7. Document why outputs are hypotheses rather than discovered truth.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, silhouette_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

customers=pd.read_csv(DATA_DIR/"customer_segments.csv")
features=customers.drop(columns="customer_id")
X=StandardScaler().fit_transform(features)
models=[]
for seed in [1,2,3,42]:
    labels=KMeans(n_clusters=4,n_init=20,random_state=seed).fit_predict(X)
    models.append(labels)
print("Silhouette:",silhouette_score(X,models[-1]))
print("Seed stability ARI:",[adjusted_rand_score(models[0],x) for x in models[1:]])
profile=features.assign(cluster=models[-1]).groupby("cluster").mean()
display(profile.round(2))

ops=pd.read_csv(DATA_DIR/"operations_anomalies.csv")
ops_features=["latency_ms","error_rate","throughput_rpm","cpu_pct","memory_pct"]
Xops=StandardScaler().fit_transform(ops[ops_features])
detector=IsolationForest(n_estimators=220,contamination=.03,random_state=42).fit(Xops)
ops["anomaly_score"]=-detector.score_samples(Xops)
print("Anomaly ranking AUC:",roc_auc_score(ops.is_anomaly,ops.anomaly_score))
display(ops.nlargest(12,"anomaly_score")[["timestamp",*ops_features,"is_anomaly","anomaly_score"]])

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.